### Data Ingestion

In [2]:
from langchain_core.documents import Document

In [6]:
doc = Document(
    page_content = "This is a random document",
    metadata = {
        "source":"randomtxt.txt",
        "page":1
    }
)
doc

Document(metadata={'source': 'randomtxt.txt', 'page': 1}, page_content='This is a random document')

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/algo_book.pdf")

docs = list(loader.lazy_load())

KeyboardInterrupt: 

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../data/algo_book.pdf")
docs = list(loader.lazy_load())

def clean_clrs_page(text: str) -> str: # text is a single page in the book
    text = re.sub(r'(\n[A-Z\-]+\(.*?\)\n)', r'\n<<<ALGO_START>>>\1', text) # pattern example: HEAP-EXTRACT-MAX(A)
    text = re.sub(r'\n\d+\s+Chapter \d+.*?\n', '\n', text) # removes page headers like 123   Chapter 6 Heapsort
    text = re.sub(r'\nProblems for Chapter \d+.*', '', text, flags=re.DOTALL) # removes the exercise questions from the page
    # NEW: remove figure captions — "Figure 6.2  The action of MAX-HEAPIFY..."
    text = re.sub(r'\nFigure \d+\.\d+.*?\n', '\n', text)
    # NEW: collapse 3+ newlines into 2 (PyPDF often over-spaces math blocks)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(
    r'\n(Theorem|Lemma|Definition|Corollary)\s+\d+\.\d+',
    r'\n<<<THEOREM_START>>>\n\1',
    text
    )
    return text

def remove_exercises(docs):
    theory_docs = []
    for doc in docs:
        page_num = doc.metadata.get("page", 0)
        # CLRS 4th ed: theory is roughly pages 28–1246, skip front/back matter
        if page_num < 28 or page_num > 1246:
            continue
        content = doc.page_content
        exercise_split = re.split(r'\n(?=Exercises\s+\d+\.\d+\s*\n)', content) # splits and returns an array where exercise_split[0] is the theory content and exercise_split[1] is the exercises content, handled inline exercise references within theory, ignoring them
        doc.page_content = clean_clrs_page(exercise_split[0]) # clean the theory content and update the document
        if len(doc.page_content.strip()) > 100: # only keep pages with substantial theory content (arbitrary threshold)
            theory_docs.append(doc)
    return theory_docs

clean_docs = remove_exercises(docs)
print(f"Pages after cleaning: {len(clean_docs)}")


Pages after cleaning: 1213


In [3]:
# ── OPEN SOURCE TOKEN LENGTH FUNCTION ─────────────────────────────────────
# QWEN Embedding model — best open source model for RAG embeddings
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B") #downloads and loads the tokenizer associated with this particular model

def token_length(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False)) # the encode function returns the token Ids List like [101, 102, 103, ...] that correspond to the and we take the length of that list to get the token count

# ── CHUNK ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,        # 512 tokens ≈ full theorem + proof in CLRS
    chunk_overlap=80,      # overlap to preserve reasoning context
    separators=[
        "\n<<<ALGO_START>>>", # custom separator for algorithm starts to keep them intact
        "\n<<<THEOREM_START>>>", # custom separator for theorem/lemma/definition starts to keep them intact
        "\n\n\n",          # section breaks
        "\n\n",            # paragraph breaks
        "\n",              # line breaks (preserves pseudocode structure)
        ". ",              # sentence boundary
        " ",               # word boundary
        "",                # character fallback (last resort)
    ],
    length_function=token_length,
)

chunks = splitter.split_documents(clean_docs)
print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i}:")
    print(f"  Tokens : {token_length(chunk.page_content)}")
    print(f"  Page   : {chunk.metadata.get('page')}")
    print(f"  Preview: {chunk.page_content[:80].strip()}...")

Total chunks: 1484

Chunk 0:
  Tokens : 468
  Page   : 28
  Preview: Output: A permutation (reordering) 
  of the input sequence
such that 
 .
Thus,...

Chunk 1:
  Tokens : 402
  Page   : 29
  Preview: Sorting is by no means the only computational problem for which
algorithms have...

Chunk 2:
  Tokens : 448
  Page   : 30
  Preview: (covered in Chapter 31), which are based on numerical algorithms
and number theo...

Chunk 3:
  Tokens : 436
  Page   : 31
  Preview: n! denotes the factorial function. Because the factorial function
grows faster t...

Chunk 4:
  Tokens : 401
  Page   : 32
  Preview: rail network because taking shorter paths results in lower labor
and fuel costs....


### Embedding the chunked tokens to vectors

In [6]:
import os
import torch # used for hardware detection
from langchain_huggingface import HuggingFaceEmbeddings

class EmbeddingManager:
    """Responsible for converting text/chunks into embeddings using Qwen3."""
    
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-Embedding-0.6B",
        batch_size: int = 32, # defines how many batches to embed at once, larger batch sizes can speed up embedding but require more GPU memory, adjust based on your hardware capabilities
        device: str = None,
    ):
        self.batch_size = batch_size
        if device:
            self.device = device
        elif torch.backends.mps.is_available():
            self.device = "mps"
        elif torch.cuda.is_available():
            self.device = "cuda"
        else:
            self.device = "cpu"
        
        print(f"Loading embedding model on {self.device}...")
        self.model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": self.device},
            encode_kwargs={
                "normalize_embeddings": True,  # required for Qwen3, divides the vectors by their length, useful for cosine similarity in FAISS
                "batch_size": self.batch_size,
            }
        )

    def embed_chunks(self, chunks: list) -> tuple[list, list]:
        """
        Convert document chunks into embeddings.
        Returns (texts, embeddings) — keeps them paired for FAISS ingestion.
        """
        print(f"Embedding {len(chunks)} chunks on {self.device}...")
        texts = [chunk.page_content for chunk in chunks]
        embeddings = self.model.embed_documents(texts)
        print(f"Done — {len(embeddings)} embeddings of dim {len(embeddings[0])}")
        return chunks, embeddings

    def embed_query(self, query: str) -> list[float]:
        """Convert a single query string into an embedding for retrieval."""
        return self.model.embed_query(query)

In [9]:
import os
from langchain_community.vectorstores import FAISS

class VectorStoreManager:
    """Responsible for storing, persisting, and retrieving from FAISS index."""

    def __init__(self, index_path: str = "../data/clrs_faiss_index"):
        self.index_path = index_path
        self.vectorstore = None

    def build(self, chunks, embeddings):

        print("Building FAISS index...")

        texts = [chunk.page_content for chunk in chunks]
        metadatas = [chunk.metadata for chunk in chunks]

        pairs = list(zip(texts, embeddings))

        self.vectorstore = FAISS.from_embeddings(
            pairs,
            embedding=None,
            metadatas=metadatas
        )

        os.makedirs(os.path.dirname(self.index_path), exist_ok=True)

        self.vectorstore.save_local(self.index_path)

        print(f"Index saved → {self.index_path} ({self.total_vectors} vectors)")

    def load(self, embedding_manager: EmbeddingManager) -> None:
        """Load persisted FAISS index from disk."""
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(
                f"No index at {self.index_path}. Call build() first."
            )
        self.vectorstore = FAISS.load_local( # loads index from disk created from build
            self.index_path,
            embeddings=embedding_manager.model, # re-instantiate the embedding model to ensure compatibility with the loaded index, especially important if using a custom or updated model
            allow_dangerous_deserialization=True
        )
        print(f"Loaded FAISS index — {self.total_vectors} vectors")

    def get_or_build(self, embedding_manager: EmbeddingManager, chunks: list = None) -> None:
        """Load index if it exists, otherwise build from chunks."""
        if os.path.exists(self.index_path):
            self.load(embedding_manager)
        elif chunks is not None:
            chunks, embeddings = embedding_manager.embed_chunks(chunks)
            self.build(chunks, embeddings)
        else:
            raise ValueError("No index found and no chunks provided to build one.")

    def search(self, query_embedding: list[float], k: int = 4) -> list:
        """Retrieve top-k chunks by a pre-computed query embedding."""
        self._guard()
        return self.vectorstore.similarity_search_by_vector(query_embedding, k=k)

    def search_with_score(self, query_embedding: list[float], k: int = 4) -> list:
        """Retrieve top-k chunks with L2 distance scores (lower = more similar)."""
        self._guard()
        return self.vectorstore.similarity_search_by_vector_with_relevance_scores( # returns a list of tuples [(Document, score), ...] where score is the L2 distance between the query embedding and the chunk embedding, lower scores indicate higher similarity
            query_embedding, k=k
        )

    @property
    def total_vectors(self) -> int:
        return self.vectorstore.index.ntotal if self.vectorstore else 0

    def _guard(self) -> None:
        if self.vectorstore is None:
            raise RuntimeError("No vectorstore loaded. Call get_or_build() first.")

In [10]:
embedder = EmbeddingManager()
store = VectorStoreManager(index_path="../data/clrs_faiss_index")

# Build once, load on every subsequent run
store.get_or_build(embedder, chunks)

# Query — embedding concern stays in EmbeddingManager
query = "how does heapsort maintain the heap property?"
query_embedding = embedder.embed_query(query)

# Retrieval concern stays in VectorStoreManager
results = store.search(query_embedding, k=4)
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print("Page:", doc.metadata.get("page"))
    print("Text:", doc.page_content[:300])

Loading embedding model on mps...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding 1484 chunks on mps...


KeyboardInterrupt: 